# Diagnostico de separadores em discursos antigos

Este caderno analisa marcas de separacao, cabecalhos editoriais, linhas de asteriscos e marcas parenteticas em discursos historicos nos Parquets `textos_parlamentares/v1`.

A execucao e read-only em relacao aos Parquets de entrada. As saidas ficam em `processed/audits/separadores_antigos/{run_id}/` no Google Drive.


## 1. Montar Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Ambiente fora do Colab; usando caminhos locais quando existirem.")


## 2. Preparar repositorio

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.

if Path("/content").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
        if not REPO_REF:
            subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    if REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
    os.chdir(REPO_DIR)
else:
    REPO_DIR = Path.cwd()
    print("Usando repositorio local:", REPO_DIR)

print("Repositorio:", Path.cwd())
subprocess.run(["git", "status", "--short"], check=False)


## 3. Instalar dependencias

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pyarrow>=15,<23", "pandas>=2,<3"],
    check=True,
)


## 4. Parametros

Por default, o caderno compara discursos anteriores a 2010 com uma faixa curta de referencia entre 2010 e 2012. Ajuste `FAIXAS` se quiser outro recorte.


In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data") if Path("/content/drive").exists() else Path("data/samples")
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)

if DATA_ROOT.name == "samples":
    PARQUET_ROOT = DATA_ROOT / "textos_parlamentares" / "v1" / "parquet"
    AUDIT_BASE = DATA_ROOT / "textos_parlamentares" / "v1" / "audits" / "separadores_antigos"
else:
    PARQUET_ROOT = DATA_ROOT / "processed" / "textos_parlamentares" / "v1" / "parquet"
    AUDIT_BASE = DATA_ROOT / "processed" / "audits" / "separadores_antigos"

RUN_ID = "separadores-antigos-discursos-" + datetime.now(timezone.utc).strftime("%Y%m%d")
OUTPUT_ROOT = AUDIT_BASE / RUN_ID

DATASETS = [
    "senado/plenario_discursos",
    "senado/congresso_discursos",
    "camara/plenario_discursos",
]

FAIXAS = {
    "historico_pre_2010": {"ano_min": None, "ano_max": 2009},
    "referencia_2010_2012": {"ano_min": 2010, "ano_max": 2012},
}

CONTEXT_CHARS = 500
MAX_EXAMPLES_PER_SEPARATOR = 50
BATCH_SIZE = 2_000
OVERWRITE = True

print("DATA_ROOT=", DATA_ROOT)
print("PARQUET_ROOT=", PARQUET_ROOT)
print("OUTPUT_ROOT=", OUTPUT_ROOT)
print("DATASETS=", DATASETS)
print("FAIXAS=", FAIXAS)


## 5. Conferir Parquets disponiveis

In [ ]:
parquets = sorted(PARQUET_ROOT.glob("*.parquet"))
print("Parquets encontrados:", len(parquets))
for path in parquets:
    print("-", path.name)
if not parquets:
    raise FileNotFoundError(f"Nenhum Parquet encontrado em {PARQUET_ROOT}")


## 6. Funcoes de diagnostico

In [ ]:
import json
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

from processamento.inventario_separadores import (
    READ_COLUMNS,
    detect_parenthetical_lines,
    detect_separator_candidates,
)


def source_dataset_from_filename(path: Path):
    stem = path.stem
    if "__" not in stem:
        return None, None
    source, dataset = stem.split("__", 1)
    return source, dataset


def faixa_for_ano(ano):
    if ano is None:
        return None
    for faixa, limits in FAIXAS.items():
        ano_min = limits.get("ano_min")
        ano_max = limits.get("ano_max")
        if ano_min is not None and ano < int(ano_min):
            continue
        if ano_max is not None and ano > int(ano_max):
            continue
        return faixa
    return None


def parse_ano(value):
    if value is None:
        return None
    try:
        return int(str(value)[:4])
    except ValueError:
        return None


def normalize_row(row, fallback_source=None, fallback_dataset=None):
    source = str(row.get("source") or fallback_source or "")
    dataset = str(row.get("dataset") or fallback_dataset or "")
    ano_int = parse_ano(row.get("ano") or row.get("data"))
    return {
        "texto_id": str(row.get("texto_id") or ""),
        "source": source,
        "dataset": dataset,
        "ano": str(ano_int) if ano_int is not None else "",
        "ano_int": ano_int,
        "mes": str(row.get("mes") or ""),
        "data": str(row.get("data") or ""),
        "documento_tipo": str(row.get("documento_tipo") or ""),
        "unidade_analitica": str(row.get("unidade_analitica") or ""),
        "texto": row.get("texto") or "",
    }


def iter_filtered_rows():
    dataset_filter = set(DATASETS)
    for parquet_path in sorted(PARQUET_ROOT.glob("*.parquet")):
        fallback_source, fallback_dataset = source_dataset_from_filename(parquet_path)
        if fallback_source and fallback_dataset and f"{fallback_source}/{fallback_dataset}" not in dataset_filter:
            continue
        parquet_file = pq.ParquetFile(parquet_path)
        available = set(parquet_file.schema_arrow.names)
        if "texto" not in available:
            continue
        columns = [column for column in READ_COLUMNS if column in available]
        for batch in parquet_file.iter_batches(batch_size=BATCH_SIZE, columns=columns):
            for raw_row in batch.to_pylist():
                row = normalize_row(raw_row, fallback_source=fallback_source, fallback_dataset=fallback_dataset)
                if f"{row['source']}/{row['dataset']}" not in dataset_filter:
                    continue
                faixa = faixa_for_ano(row["ano_int"])
                if faixa is None:
                    continue
                if not row["texto"]:
                    continue
                row["faixa"] = faixa
                yield row


def prepare_output_root():
    expected_base = AUDIT_BASE.resolve(strict=False)
    resolved_output = OUTPUT_ROOT.resolve(strict=False)
    if OVERWRITE and OUTPUT_ROOT.exists():
        try:
            resolved_output.relative_to(expected_base)
        except ValueError as exc:
            raise ValueError(f"Recusa apagar saida fora de {expected_base}: {resolved_output}") from exc
        shutil.rmtree(OUTPUT_ROOT)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## 7. Executar diagnostico

In [ ]:
prepare_output_root()

coverage = Counter()
separator_counts = Counter()
separator_text_ids = defaultdict(set)
separator_examples = {}
parenthetical_counts = Counter()
parenthetical_text_ids = defaultdict(set)
parenthetical_examples = {}
examples = []
example_counts = Counter()
processed_rows = 0

for row in iter_filtered_rows():
    processed_rows += 1
    coverage[(row["faixa"], row["source"], row["dataset"], row["ano"])] += 1

    for candidate in detect_separator_candidates(row, context_chars=CONTEXT_CHARS):
        candidate["faixa"] = row["faixa"]
        key = (
            row["faixa"],
            candidate["source"],
            candidate["dataset"],
            candidate["ano"],
            candidate["action"],
            candidate["kind"],
            candidate["separator_normalized"],
        )
        separator_counts[key] += 1
        separator_text_ids[key].add(candidate["texto_id"])
        separator_examples.setdefault(key, candidate["separator_text"])
        example_key = (row["faixa"], candidate["source"], candidate["dataset"], candidate["action"], candidate["separator_normalized"])
        if example_counts[example_key] < MAX_EXAMPLES_PER_SEPARATOR:
            examples.append(candidate)
            example_counts[example_key] += 1

    for parenthetical in detect_parenthetical_lines(row):
        parenthetical["faixa"] = row["faixa"]
        key = (
            row["faixa"],
            parenthetical["source"],
            parenthetical["dataset"],
            parenthetical["ano"],
            parenthetical["parenthetical_normalized"],
        )
        parenthetical_counts[key] += 1
        parenthetical_text_ids[key].add(parenthetical["texto_id"])
        parenthetical_examples.setdefault(key, parenthetical["parenthetical_text"])

coverage_rows = [
    {"faixa": faixa, "source": source, "dataset": dataset, "ano": ano, "textos": count}
    for (faixa, source, dataset, ano), count in coverage.items()
]
separator_rows = [
    {
        "faixa": faixa,
        "source": source,
        "dataset": dataset,
        "ano": ano,
        "action": action,
        "kind": kind,
        "separator_normalized": normalized,
        "separator_example": separator_examples.get(key, ""),
        "occurrences": count,
        "textos": len(separator_text_ids.get(key, set())),
    }
    for key, count in separator_counts.items()
    for faixa, source, dataset, ano, action, kind, normalized in [key]
]
parenthetical_rows = [
    {
        "faixa": faixa,
        "source": source,
        "dataset": dataset,
        "ano": ano,
        "action": "keep",
        "parenthetical_normalized": normalized,
        "parenthetical_example": parenthetical_examples.get(key, ""),
        "occurrences": count,
        "textos": len(parenthetical_text_ids.get(key, set())),
    }
    for key, count in parenthetical_counts.items()
    for faixa, source, dataset, ano, normalized in [key]
]

def sorted_frame(rows, columns, sort_cols, ascending=True):
    frame = pd.DataFrame(rows, columns=columns)
    if frame.empty:
        return frame
    return frame.sort_values(sort_cols, ascending=ascending)


coverage_df = sorted_frame(
    coverage_rows,
    ["faixa", "source", "dataset", "ano", "textos"],
    ["faixa", "source", "dataset", "ano"],
)
separators_df = sorted_frame(
    separator_rows,
    ["faixa", "source", "dataset", "ano", "action", "kind", "separator_normalized", "separator_example", "occurrences", "textos"],
    ["faixa", "source", "dataset", "ano", "action", "kind", "occurrences"],
    ascending=[True, True, True, True, True, True, False],
)
parentheticals_df = sorted_frame(
    parenthetical_rows,
    ["faixa", "source", "dataset", "ano", "action", "parenthetical_normalized", "parenthetical_example", "occurrences", "textos"],
    ["faixa", "source", "dataset", "ano", "occurrences"],
    ascending=[True, True, True, True, False],
)

coverage_path = OUTPUT_ROOT / "cobertura_discursos_antigos.csv"
separators_path = OUTPUT_ROOT / "separadores_antigos_resumo.csv"
examples_path = OUTPUT_ROOT / "separadores_antigos_exemplos.jsonl"
parentheticals_path = OUTPUT_ROOT / "parenteticos_antigos_resumo.csv"
manifest_path = OUTPUT_ROOT / "manifest.json"

coverage_df.to_csv(coverage_path, index=False)
separators_df.to_csv(separators_path, index=False)
parentheticals_df.to_csv(parentheticals_path, index=False)
with examples_path.open("w", encoding="utf-8") as handle:
    for example in examples:
        handle.write(json.dumps(example, ensure_ascii=False, sort_keys=True) + "\n")

manifest = {
    "run_id": RUN_ID,
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "parquet_root": str(PARQUET_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "datasets": DATASETS,
    "faixas": FAIXAS,
    "context_chars": CONTEXT_CHARS,
    "max_examples_per_separator": MAX_EXAMPLES_PER_SEPARATOR,
    "batch_size": BATCH_SIZE,
    "processed_rows": processed_rows,
    "separator_occurrences": int(sum(separator_counts.values())),
    "parenthetical_occurrences": int(sum(parenthetical_counts.values())),
    "example_count": len(examples),
    "output_files": {
        "cobertura_discursos_antigos": str(coverage_path),
        "separadores_antigos_resumo": str(separators_path),
        "separadores_antigos_exemplos": str(examples_path),
        "parenteticos_antigos_resumo": str(parentheticals_path),
        "manifest": str(manifest_path),
    },
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print(json.dumps(manifest, ensure_ascii=False, indent=2))


## 8. Ver cobertura anual

In [ ]:
coverage_df.head(50)


In [ ]:
if not coverage_df.empty:
    pivot = coverage_df.pivot_table(
        index=["source", "dataset", "ano"],
        columns="faixa",
        values="textos",
        aggfunc="sum",
        fill_value=0,
    ).reset_index()
    display(pivot.tail(80))
else:
    print("Sem registros nas faixas configuradas.")


## 9. Ver separadores mais frequentes

In [ ]:
if separators_df.empty:
    print("Nenhum separador candidato encontrado.")
else:
    display(separators_df.head(100))


In [ ]:
if not separators_df.empty:
    comparative = separators_df.groupby(["faixa", "source", "dataset", "action", "kind", "separator_normalized"], as_index=False)["occurrences"].sum()
    comparative = comparative.sort_values(["faixa", "source", "dataset", "occurrences"], ascending=[True, True, True, False])
    display(comparative.head(100))


## 10. Abrir exemplos com contexto

In [ ]:
def load_examples(path=examples_path):
    if not Path(path).exists():
        return pd.DataFrame()
    records = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            records.append(json.loads(line))
    return pd.DataFrame(records)

examples_df = load_examples()
print("Exemplos:", len(examples_df))
if not examples_df.empty:
    display(examples_df[["faixa", "source", "dataset", "ano", "action", "kind", "separator_normalized", "texto_id", "line_number", "trailing_chars"]].head(100))


In [ ]:
def show_example(index=0):
    if examples_df.empty:
        print("Sem exemplos carregados.")
        return
    row = examples_df.iloc[index].to_dict()
    print("texto_id:", row.get("texto_id"))
    print("faixa:", row.get("faixa"), "source/dataset:", f"{row.get('source')}/{row.get('dataset')}", "ano:", row.get("ano"))
    print("action/kind:", row.get("action"), row.get("kind"))
    print("separator:", row.get("separator_text"))
    print("\n--- contexto antes ---")
    print(row.get("context_before") or "")
    print("\n--- separador ---")
    print(row.get("separator_text") or "")
    print("\n--- contexto depois ---")
    print(row.get("context_after") or "")

show_example(0)


## 11. Marcas parenteticas mais frequentes

In [ ]:
if parentheticals_df.empty:
    print("Nenhuma marca parentetica encontrada.")
else:
    display(parentheticals_df.head(100))


## 12. Arquivos gerados

In [ ]:
print("OUTPUT_ROOT=", OUTPUT_ROOT)
for path in sorted(OUTPUT_ROOT.glob("*")):
    print(path.name, path.stat().st_size, "bytes")
